In [ ]:
code = 'SUT_PSL'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SUT_per_minute_mtm(bt, start_time, end_time, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om, seperate=False):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        entry_time = start_dt
        std_sl_time, std_mtm_data = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, per_minute_mtm=True)
        
        ut, ut_mtm_data = '', pd.Series()
        if std_sl_time and (std_sl_time < end_dt - datetime.timedelta(minutes=5)):
            
            ce_inc_rate = (bt.options_data.loc[(std_sl_time, ce_scrip), 'close'] - ce_price)/ce_price
            pe_inc_rate = (bt.options_data.loc[(std_sl_time, pe_scrip), 'close'] - pe_price)/pe_price
            
            if ce_inc_rate > pe_inc_rate:
                ut = 'PE' if ut_orderside == 'SELL' else 'CE'
            elif ce_inc_rate < pe_inc_rate:
                ut = 'CE' if ut_orderside == 'SELL' else 'PE'

            if ut:
                ut_scrip, ut_price, ut_future_price, ut_start_dt = bt.get_strike(std_sl_time, end_dt, om=ut_om, only=ut)

                if ut_scrip:
                    from_candle_close = True if ut_method == 'CC' else False
                    ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(ut_start_dt, end_dt, ut_scrip, sl=ut_sl, orderside=ut_orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

        if seperate:
            return std_mtm_data, ut_mtm_data
        else:
            std_mtm_data = set_pm_time_index(std_mtm_data, time_index)

            if ut:
                ut_mtm_data = set_pm_time_index(ut_mtm_data, time_index)
                return std_mtm_data+ut_mtm_data
            else:
                return std_mtm_data

    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om])
        return

In [ ]:
def SUT_PSL(bt, start_time, end_time, last_trade_time, trade_interval, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        last_trade_dt = datetime.datetime.combine(bt.current_date, last_trade_time)

        entry_time = start_dt
        time_range = pd.date_range(start_dt, last_trade_dt, freq=trade_interval.lower()).time
        
        per_minute_trades = [SUT_per_minute_mtm(bt, re_time, end_time, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om) for re_time in time_range]
        per_minute_trades = [t for t in per_minute_trades if t is not None]
        per_minute_mtm = np.sum(per_minute_trades, axis=0)
        
        if only_mtm:
            mtm_time_list = list(per_minute_mtm)
            mtm_time_list += [mtm_time_list[-1]]
        else:
            total_minutes = len(time_range)
            per_minute_margin = int(fund/total_minutes)
            shares_per_minute = int(per_minute_margin/margin_per_share)
            per_minute_mtm_total = per_minute_mtm * shares_per_minute

            mtm_dict ={}
            for mtm_percent, check_mtm in check_mtms.items():

                if check_mtm > 0:
                    condition = np.where(per_minute_mtm_total > check_mtm)[0]
                else:
                    condition = np.where(per_minute_mtm_total < check_mtm)[0]

                try:
                    idx = condition[0]
                    time = time_index[idx]
                    mtm = per_minute_mtm_total[idx+1]
                except:
                    time, mtm = '', per_minute_mtm_total[-1]

                mtm_dict[mtm_percent] = (time, mtm)

            mtm_time_list = [fund] + [item for value in mtm_dict.values() for item in value]
        
        return [code, bt.index, start_time, end_time, last_trade_time, trade_interval, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + mtm_time_list
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, last_trade_time, trade_interval, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_LastTradeTime/P_TradeInterval/P_OrderSide/P_SL/P_intraSL/P_OM/P_UTOrderSide/P_UTMethod/P_UTSL/P_UTOM/Date/Day/DTE/EntryTime/Future/Fund')
            
            only_mtm = False
            if only_mtm:
                log_time_col = get_pm_time_index(datetime.datetime.now()).time
                log_cols += '/'.join(map(str, log_time_col))
                log_cols += '/Final.PNL'
            else:
                notinal_value = 12
                fund = 100_00_00_00
                mtm_percent_stop = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1, -1.25, -1.5, -1.75, -2, -2.5, -3, -3.5, -4, -100, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 1.75, 2, 2.5, 3, 3.5, 4]
                log_cols += 'Fund'
                for mtmp in mtm_percent_stop:
                    log_cols += f'/{mtmp:.2f}.Time/{mtmp:.2f}.PNL'

            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")
                    
                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    time_index = get_pm_time_index(bt.current_date)
                    future_price = bt.future_data['close'].iloc[0]
                    
                    if not only_mtm:
                        margin_per_share = future_price * (notinal_value / 100)
                        check_mtms = {mtm_percent: fund * mtm_percent / 100 for mtm_percent in mtm_percent_stop}
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SUT_PSL(bt, row.entry_time, row.exit_time, row.last_trade_time, row.trade_interval, row.orderside, row.sl, row.intra_sl, row.om, row.ut_orderside, row.ut_method, row.ut_sl, row.ut_om) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))